In [1]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

import os
os.environ["HF_HOME"] = "/shared/data3/pk36/.cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"

import argparse
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from dataclasses import dataclass
import json_repair
import re
import json
from typing import List, Dict
import time

from search import search_semantic_scholar, collect_snippets
from prompts import (
    create_initial_decomposition_prompt,
    create_target_domain_analysis_prompt,
    create_cross_domain_query_prompt,
    create_cross_domain_analysis_prompt,
    initial_decomposition_schema,
    target_domain_analysis_schema,
    cross_domain_queries_schema,
    cross_domain_analysis_schema
)
from classes import ResearchProblem, Question, Domain
from utils import prepare_output
from main import batch_llm_inference, retrieve_papers_for_question

/home/pk36/structured_survey/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-19 23:23:55 [__init__.py:216] Automatically detected platform cuda.


## Setup

In [16]:
@dataclass
class Args:
    problem_file: str = "data/mimo.txt"
    target_domain: str = "Engineering"
    model_name: str = "Qwen/Qwen3-8B"
    output_dir: str = "output_debug"
    max_papers_per_query: int = 20

args = Args()

In [17]:
# Read problem statement
if os.path.exists(args.problem_file):
    with open(args.problem_file, "r") as f:
        problem_file_text = f.read()
        match = re.search(r"Problem Statement:\s*(.*)", problem_file_text)
        if match:
            problem_statement = match.group(1).strip()
        else:
            print("Could not find problem statement in file!")
    print(f"Problem Statement: {problem_statement}\n")
else:
    print(f"File {args.problem_file} does not exist!")

# Create output file path
output_file_name = os.path.splitext(os.path.basename(args.problem_file))[0] + f"_{args.max_papers_per_query}_results.json"
condensed_output_file_name = os.path.splitext(os.path.basename(args.problem_file))[0] + f"_{args.max_papers_per_query}_condensed.json"

args.output_file = os.path.join(args.output_dir, output_file_name)
args.condensed_output_file = os.path.join(args.output_dir, condensed_output_file_name)

# Create output directory if needed
os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

Problem Statement: How do we reduce the energy consumption of baseband processing in future massive MIMO systems?



In [4]:
# Initialize vLLM model
print("Loading model...")
llm = LLM(model=args.model_name, tensor_parallel_size=2)
print("Model loaded.\n")

Loading model...
INFO 01-19 23:24:02 [utils.py:233] non-default args: {'tensor_parallel_size': 2, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}
INFO 01-19 23:24:02 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 01-19 23:24:02 [model.py:1510] Using max model len 40960


2026-01-19 23:24:02,859	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-19 23:24:02 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=2918025) INFO 01-19 23:24:03 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=2918025) INFO 01-19 23:24:03 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=ObservabilityConfig(show_hidde

...

(EngineCore_DP0 pid=2918025) (Worker_TP0 pid=2918051) INFO 01-19 23:24:12 [default_loader.py:267] Loading weights took 2.76 seconds
(EngineCore_DP0 pid=2918025) (Worker_TP1 pid=2918057) INFO 01-19 23:24:12 [gpu_model_runner.py:2653] Model loading took 7.6394 GiB and 3.293028 seconds
(EngineCore_DP0 pid=2918025) (Worker_TP0 pid=2918051) INFO 01-19 23:24:12 [gpu_model_runner.py:2653] Model loading took 7.6394 GiB and 3.503301 seconds
(EngineCore_DP0 pid=2918025) (EngineCore_DP0 pid=2918025) (Worker_TP1 pid=2918057) (Worker_TP0 pid=2918051) INFO 01-19 23:24:20 [backends.py:548] Using cache directory: /home/pk36/.cache/vllm/torch_compile_cache/8f92c1d62e/rank_1_0/backbone for vLLM's torch.compile
INFO 01-19 23:24:20 [backends.py:548] Using cache directory: /home/pk36/.cache/vllm/torch_compile_cache/8f92c1d62e/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2918025) (EngineCore_DP0 pid=2918025) (Worker_TP1 pid=2918057) (Worker_TP0 pid=2918051) INFO 01-19 23:24:20 [backends.py

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:04<00:00, 16.13it/s]
Capturing CUDA graphs (decode, FULL):  46%|████▌     | 16/35 [00:01<00:01, 16.70it/s]

(EngineCore_DP0 pid=2918025) (Worker_TP1 pid=2918057) INFO 01-19 23:24:34 [custom_all_reduce.py:203] Registering 7446 cuda graph addresses


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.42it/s]


(EngineCore_DP0 pid=2918025) (Worker_TP0 pid=2918051) INFO 01-19 23:24:35 [custom_all_reduce.py:203] Registering 7446 cuda graph addresses
(EngineCore_DP0 pid=2918025) (Worker_TP1 pid=2918057) INFO 01-19 23:24:35 [gpu_model_runner.py:3480] Graph capturing finished in 8 secs, took 0.88 GiB
(EngineCore_DP0 pid=2918025) (Worker_TP0 pid=2918051) INFO 01-19 23:24:35 [gpu_model_runner.py:3480] Graph capturing finished in 8 secs, took 0.88 GiB
(EngineCore_DP0 pid=2918025) INFO 01-19 23:24:35 [core.py:210] init engine (profile, create kv cache, warmup model) took 23.00 seconds
INFO 01-19 23:24:37 [llm.py:306] Supported_tasks: ['generate']
Model loaded.



## Decomposition

In [25]:
prompt = create_initial_decomposition_prompt(problem_statement, args.target_domain)
messages = [{"role": "user", "content": prompt}]

decomposition_outputs = batch_llm_inference(
    llm, 
    [messages], 
    initial_decomposition_schema,
    temperature=0.7
)
decomposition_output = decomposition_outputs[0]

if decomposition_output is None:
    print("Failed to get decomposition output!")

Processed prompts: 100%|██████████| 1/1 [00:08<00:00,  8.45s/it, est. speed input: 65.78 toks/s, output: 69.10 toks/s]


In [26]:
# Create ResearchProblem object
research_problem = ResearchProblem.from_initial_decomposition(
    decomposition_output, 
    args.target_domain
)

print(f"Generated {len(research_problem.research_questions)} research questions:")
for q in research_problem.research_questions:
    print(f"  - {q.id}: {q.question}")
print()

Generated 5 research questions:
  - q1: How can we optimize signal processing algorithms to minimize computational energy use?
  - q2: What are the key hardware constraints limiting energy efficiency in baseband units?
  - q3: How can we dynamically adjust processing resources based on system load?
  - q4: What are the most effective techniques for reducing data transmission overhead in massive MIMO systems?
  - q5: How can machine learning be used to predict and optimize baseband energy use?



In [24]:
print(json.dumps(decomposition_output, indent=2))

{
  "problem_statement": "How do we reduce the energy consumption of baseband processing in future massive MIMO systems?",
  "target_domain": "Engineering",
  "fine_grained_domain": "Wireless Communications",
  "core_challenge": "Massive MIMO systems require high computational power for baseband processing, leading to significant energy consumption. Reducing this energy use is critical for improving the sustainability and efficiency of next-generation wireless networks.",
  "research_questions": [
    {
      "id": "q1",
      "question": "How can we optimize signal processing algorithms to minimize energy use?",
      "rationale": "Efficient algorithm design is a foundational challenge in reducing energy consumption. Optimizing processing steps directly impacts power efficiency without compromising system performance.",
      "target_domain_queries": [
        "energy efficient signal processing",
        "low power algorithm design",
        "compute optimization for baseband"
      

## Target Domain Analysis

In [27]:
# Step 2a: Retrieve papers for all questions in target domain
print("\n2a. Retrieving papers from target domain...")
for question in research_problem.research_questions:
    print(f"  Retrieving for {question.id}...")
    papers = retrieve_papers_for_question(
        question, 
        research_problem.target_domain,
        max_papers=args.max_papers_per_query
    )
    research_problem.target_domain.add_question_papers(question, papers)
    print(f"    -Retrieved {len(papers)} papers")


2a. Retrieving papers from target domain...
  Retrieving for q1...
	 -Searching Semantic Scholar for query: energy-efficient signal processing in domain: Engineering.
	 -Searching Semantic Scholar for query: low-power algorithm design in domain: Engineering.
	 -Searching Semantic Scholar for query: computational energy trade-offs in domain: Engineering.
    -Retrieved 15 papers
  Retrieving for q2...
	 -Searching Semantic Scholar for query: baseband power constraints in domain: Engineering.
	 -Searching Semantic Scholar for query: hardware energy efficiency in domain: Engineering.
	 -Searching Semantic Scholar for query: thermal management in baseband in domain: Engineering.
    -Retrieved 13 papers
  Retrieving for q3...
	 -Searching Semantic Scholar for query: dynamic resource allocation in domain: Engineering.
	 -Searching Semantic Scholar for query: load-aware energy management in domain: Engineering.
	 -Searching Semantic Scholar for query: adaptive baseband processing in domain:

In [28]:
# Step 2b: Batch analyze all questions in target domain
print("\n2b. Analyzing target domain papers (batch inference)...")

# Prepare batch of analysis prompts
analysis_messages_list = []
for question in research_problem.research_questions:
    papers = research_problem.target_domain.fetch_question_papers(question)
    
    if not papers:
        print(f"  Warning: No papers for {question.id}, skipping analysis")
        continue
    
    prompt = create_target_domain_analysis_prompt(
        research_problem=research_problem.problem_statement,
        question=question.question,
        question_rationale=question.rationale,
        papers_with_snippets=papers,
        target_domain=args.target_domain
    )
    messages = [{"role": "user", "content": prompt}]
    analysis_messages_list.append(messages)

# Batch inference for all analyses
if analysis_messages_list:
    analysis_outputs = batch_llm_inference(
        llm,
        analysis_messages_list,
        target_domain_analysis_schema,
        temperature=0.5  # Lower temperature for analysis
    )


2b. Analyzing target domain papers (batch inference)...


Processed prompts: 100%|██████████| 5/5 [00:34<00:00,  6.91s/it, est. speed input: 665.23 toks/s, output: 228.10 toks/s]


In [29]:
# Process analysis results
for i, (question, analysis_output) in enumerate(zip(research_problem.research_questions, analysis_outputs)):
    if analysis_output is None:
        print(f"  Failed to analyze {question.id}")
        continue
    
    question.target_domain_analysis = analysis_output
    research_problem.target_domain.add_question_analysis(question, analysis_output)
    
    # Determine if addressed
    assessment = analysis_output.get("overall_assessment", "largely unaddressed").lower()
    is_addressed = "substantially" in assessment or "partial" in assessment
    question.mark_as_addressed(is_addressed)
    
    print(f"  {question.id}: {assessment}")
    
    # Create sub-questions for remaining challenges
    remaining_challenges = analysis_output.get("remaining_challenges", [])
    for challenge_data in remaining_challenges:
        challenge = research_problem.add_remaining_challenge(question, challenge_data)
        print(f"    -> New challenge: {challenge.question}")

  q1: partially addressed
    -> New challenge: How can we achieve energy efficiency in signal processing algorithms without sacrificing critical performance or accuracy?
    -> New challenge: How can we generalize energy-efficient signal processing techniques across different hardware platforms and application domains?
    -> New challenge: How can we ensure real-time energy efficiency in dynamic and resource-constrained environments?
  q2: partially addressed
    -> New challenge: How can we fundamentally reduce the power consumption of baseband units without compromising performance or introducing new thermal bottlenecks?
    -> New challenge: What are the long-term material and manufacturing limitations that prevent energy-proportional baseband units from being widely adopted?
    -> New challenge: How can we design baseband units that adapt dynamically to varying operational conditions, such as temperature and traffic load, to maintain energy efficiency?
  q3: partially addressed


## Cross-Domain Query Generation

In [30]:
# Get all questions needing cross-domain search
questions_needing_cross_domain = research_problem.get_questions_needing_cross_domain()

print(f"\nFound {len(questions_needing_cross_domain)} questions needing cross-domain search:")
for q in questions_needing_cross_domain:
    print(f"  - {q.id}: {q.question}")

if not questions_needing_cross_domain:
    print("\nAll questions addressed in target domain! No cross-domain search needed.")
else:
    # Step 3a: Generate cross-domain queries (batch)
    print("\n3a. Generating cross-domain queries (batch inference)...")
    
    cross_domain_messages_list = []
    for question in questions_needing_cross_domain:
        # Get target domain assessment if available
        target_assessment = None
        if question.parent_question and question.parent_question.target_domain_analysis:
            # This is a remaining challenge (includes )
            target_assessment = question.rationale
        elif question.target_domain_analysis:
            # This is an original question (iterate over all challenges in target domain analysis)
            target_assessment = ""
            for challenge in question.remaining_challenges:
                target_assessment += f"- {challenge.rationale}\n"
        
        prompt = create_cross_domain_query_prompt(
            problem_statement=research_problem.problem_statement,
            question=question.question,
            question_rationale=question.rationale,
            target_domain=args.target_domain,
            target_domain_assessment=target_assessment
        )
        messages = [{"role": "user", "content": prompt}]
        cross_domain_messages_list.append(messages)
    
    # Batch inference for cross-domain queries
    cross_domain_outputs = batch_llm_inference(
        llm,
        cross_domain_messages_list,
        cross_domain_queries_schema,
        temperature=0.7
    )


Found 15 questions needing cross-domain search:
  - c1: How can we achieve energy efficiency in signal processing algorithms without sacrificing critical performance or accuracy?
  - c2: How can we generalize energy-efficient signal processing techniques across different hardware platforms and application domains?
  - c3: How can we ensure real-time energy efficiency in dynamic and resource-constrained environments?
  - c1: How can we fundamentally reduce the power consumption of baseband units without compromising performance or introducing new thermal bottlenecks?
  - c2: What are the long-term material and manufacturing limitations that prevent energy-proportional baseband units from being widely adopted?
  - c3: How can we design baseband units that adapt dynamically to varying operational conditions, such as temperature and traffic load, to maintain energy efficiency?
  - c1: How can we ensure real-time adaptability of baseband processing in massive MIMO systems without compromis

Processed prompts: 100%|██████████| 15/15 [00:06<00:00,  2.31it/s, est. speed input: 1764.28 toks/s, output: 581.48 toks/s]


In [31]:
# Process cross-domain query results
cross_domain_analysis_prompts = []
cross_domain_analysis_keys = []
for question, cross_domain_output in zip(questions_needing_cross_domain, cross_domain_outputs):
    if cross_domain_output is None:
        print(f"  Failed to generate cross-domain queries for {question.id}")
        continue
    
    question.cross_domain_queries = cross_domain_output
    
    print(f"\n  {question.question}:")
    for domain_search in cross_domain_output.get("cross_domain_searches", []):
        domain_name = domain_search["domain"]
        queries = domain_search["queries"]
        
        # Get or create domain
        domain = research_problem.get_or_create_domain(domain_name)
        domain.add_question_queries(question, queries)
        question.add_external_domain(domain)
        
        print(f"    - {domain_name}: {len(queries)} queries")
        papers = retrieve_papers_for_question(
            question,
            domain,
            max_papers=args.max_papers_per_query
        )
        
        # Conduct cross-domain analysis on domain papers
        cross_domain_analysis_prompt = create_cross_domain_analysis_prompt(
            problem_statement=research_problem.problem_statement,
            question=question.question,
            question_rationale=question.rationale,
            source_domain=domain_name,
            papers_with_snippets=papers,
            target_domain=research_problem.target_domain
        )
        cross_domain_analysis_messages = [{"role": "user", "content": cross_domain_analysis_prompt}]
        cross_domain_analysis_prompts.append(cross_domain_analysis_messages)
        cross_domain_analysis_keys.append((question, domain))
        
        domain.add_question_papers(question, papers)
        domain_search["retrieved_papers"] = papers
        print(f"      -Retrieved {len(papers)} papers")


  How can we achieve energy efficiency in signal processing algorithms without sacrificing critical performance or accuracy?:
    - Computer Science: 3 queries
	 -Searching Semantic Scholar for query: energy-efficient algorithm design in domain: Computer Science.
	 -Searching Semantic Scholar for query: approximate computing techniques in domain: Computer Science.
	 -Searching Semantic Scholar for query: latency-accuracy trade-offs in domain: Computer Science.
      -Retrieved 14 papers
    - Physics: 3 queries
	 -Searching Semantic Scholar for query: energy minimization in systems in domain: Physics.
	 -Searching Semantic Scholar for query: optimal control theory in domain: Physics.
	 -Searching Semantic Scholar for query: constraint-based optimization in domain: Physics.
      -Retrieved 15 papers
    - Mathematics: 3 queries
	 -Searching Semantic Scholar for query: optimization under constraints in domain: Mathematics.
	 -Searching Semantic Scholar for query: numerical accuracy tra

In [30]:
# Reconstruct prompts for cross-domain analysis based on updated prompt template
reconstructed_cross_domain_analysis_prompts = []
for (question, domain) in cross_domain_analysis_keys:
    papers = domain.fetch_question_papers(question)
    
    cross_domain_analysis_prompt = create_cross_domain_analysis_prompt(
        problem_statement=research_problem.problem_statement,
        question=question.question,
        question_rationale=question.rationale,
        source_domain=domain.domain_name,
        papers_with_snippets=papers,
        target_domain=research_problem.target_domain
    )
    cross_domain_analysis_messages = [{"role": "user", "content": cross_domain_analysis_prompt}]
    reconstructed_cross_domain_analysis_prompts.append(cross_domain_analysis_messages)

In [32]:
# Step 3b: Batch cross-domain analyses
print("\n3b. Analyzing cross-domain papers (batch inference)...")
if cross_domain_analysis_prompts:
    cross_domain_analysis_outputs = batch_llm_inference(
        llm,
        cross_domain_analysis_prompts,
        cross_domain_analysis_schema,
        temperature=0.5
    )
    
    # # Process cross-domain analysis results
    # for (question, domain), analysis_output in zip(cross_domain_analysis_keys, cross_domain_analysis_outputs):
    #     if analysis_output is None:
    #         print(f"  Failed to analyze cross-domain papers for question '{question.id}' in domain '{domain.domain_name}'")
    #         continue
        
    #     question.add_cross_domain_analysis(domain, analysis_output)
    #     print(f"  Analyzed cross-domain papers for question '{question.id}' in domain '{domain.domain_name}'")


3b. Analyzing cross-domain papers (batch inference)...


Processed prompts: 100%|██████████| 45/45 [01:58<00:00,  2.64s/it, est. speed input: 1966.85 toks/s, output: 606.58 toks/s]


In [37]:
print(reconstructed_cross_domain_analysis_prompts[0][0]['content'])

You are an expert at extracting cross-disciplinary insights. Analyze papers from an external domain to assess their relevance to a research question and extract high-level takeaways.

# RESEARCH PROBLEM
The exponential growth of scientific literature has made it increasingly challenging for researchers to identify, compare, and contextualize novel contributions across related works.

# RESEARCH QUESTION
How can systems effectively capture and contextualize the relevance of scientific contributions across diverse domains in real-time and at scale?

**Rationale**: Real-time and scalable relevance assessment is critical for filtering and comparing new work in rapidly evolving fields, which is essential for addressing the research problem. Current methods often rely on static or historical data and struggle with real-time relevance assessment, especially when domains are highly dynamic or interdisciplinary.

# SOURCE DOMAIN
Economics

# TARGET DOMAIN (for context)
Domain(domain_name=Comput

In [33]:
for idx, out in enumerate(cross_domain_analysis_outputs):
    if_relevant = [1 if p["is_relevant"] else 0 for p in out["paper_relevance"]]
    prop_relevant = sum(if_relevant)/len(if_relevant)
    print(idx, out["question"], out["source_domain"], f": {sum(if_relevant)}/{len(if_relevant)}", prop_relevant)

0 How can we achieve energy efficiency in signal processing algorithms without sacrificing critical performance or accuracy? Computer Science : 11/11 1.0
1 How can we achieve energy efficiency in signal processing algorithms without sacrificing critical performance or accuracy? Physics : 11/15 0.7333333333333333
2 How can we achieve energy efficiency in signal processing algorithms without sacrificing critical performance or accuracy? Mathematics : 8/12 0.6666666666666666
3 How can we generalize energy-efficient signal processing techniques across different hardware platforms and application domains? Computer Science : 14/15 0.9333333333333333
4 How can we generalize energy-efficient signal processing techniques across different hardware platforms and application domains? Mathematics : 3/12 0.25
5 How can we generalize energy-efficient signal processing techniques across different hardware platforms and application domains? Physics : 13/14 0.9285714285714286
6 How can we ensure real-ti

In [35]:
research_problem.fine_grained_domain

'Wireless Communications'

In [34]:
print(json.dumps(cross_domain_analysis_outputs[5], indent=2))

{
  "question": "How can we generalize energy-efficient signal processing techniques across different hardware platforms and application domains?",
  "source_domain": "Physics",
  "paper_relevance": [
    {
      "paper_title": "Towards a Theory of Conservative Computing",
      "is_relevant": true,
      "relevance_explanation": "This paper introduces a framework for energy conservation in computation, which is analogous to the goal of energy-efficient signal processing in massive MIMO systems."
    },
    {
      "paper_title": "Energy-Consumption Advantage of Quantum Computation",
      "is_relevant": true,
      "relevance_explanation": "This paper discusses energy conservation and efficiency in computation, which aligns with the goal of reducing energy consumption in signal processing."
    },
    {
      "paper_title": "Thermodynamic Perspectives on Computational Complexity: Exploring the P vs. NP Problem",
      "is_relevant": true,
      "relevance_explanation": "This paper int

In [ ]:
index = 1
focus_q = cross_domain_analysis_keys[index][0]
focus_d = cross_domain_analysis_keys[index][1]

print(json.dumps(focus_d.fetch_question_papers(focus_q), indent=2))

{
  "The Motivation Problem of Epistemic Expressivists": [
    "Suppose the attributor saw Mary looking outside and noticing the tell-tale visual signs of rain. It is possible then that the relevant procedures that the attributor endorses are procedures such as When you see raindrops outside, form the belief that it is raining outside, or perhaps more generally When you receive direct perceptual evidence of an event, form the belief that this event is happening. \n\nRidge's theory is ecumenical because it holds that the mental state expressed by an epistemic claim includes an ordinary belief (clause (II) above). In fact, many expressivist theories of epistemic discourse also satisfy this characterization (e.g., Chrisman 2007;Kappel & Moeller 2014). \n\nLet us end with three terminological notes. First, we follow common usage in employing the expression epistemic judgment to refer to the full mental state expressed by an epistemic claim. So, for example, within the context of Ridge's ac